<a href="https://colab.research.google.com/github/eshwar-2006/flyrank-ml-internship/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eshwar-2006/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/eshwar-2006/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully!
Rows: 30000
Columns: 44


In [ ]:
# Change the current working directory to the repository root
%cd flyrank-ml-internship
print(f"Changed working directory to {os.getcwd()}")

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
### Task Type: Ranking

My lane is Content Refresh / Content Opportunity Scoring.

I frame this as a ranking problem because the goal is to prioritize content pages for human review.

Instead of making only a yes/no decision, the system should produce an ordered list of pages from higher opportunity to lower opportunity.

This is useful because an editorial team has limited time and cannot review every page at once. A ranking system can help identify which pages should be examined first using measurable content and search-performance signals.

The final output supports a human decision about whether a page should be refreshed, investigated, or left unchanged.

In [ ]:
print("ML task type: Ranking")
print("Goal: Prioritize content pages for human review.")
print("Output: An ordered review queue.")

ML task type: Ranking
Goal: Prioritize content pages for human review.
Output: An ordered review queue.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# Check the columns available for defining the proxy

print("Columns available for the proxy:")

possible_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction"
]

for col in possible_columns:
    if col in df.columns:
        print("✓", col)
    else:
        print("✗", col, "- not found")

Columns available for the proxy:
✓ content_id
✓ impressions_90d
✓ clicks_90d
✓ ctr
✓ avg_position
✓ content_age_days
✓ days_since_last_update
✓ trend_direction


### Target / Proxy

The ideal target would be the future improvement in content performance after a refresh.

However, that outcome is not directly available in the current dataset because we do not have a controlled before-and-after experiment for every page.

Therefore, I will use an observable proxy based on current search-performance signals.

A useful proxy is whether a page is showing a declining trend while still receiving measurable search impressions.

This proxy is used to identify pages that may deserve further investigation.

The proxy is a decision-support signal, not proof that refreshing a page will improve its future performance.

### Target / Proxy

The ideal target would be whether a content page is genuinely under-capturing clicks relative to its expected potential, considering factors such as search position and visibility.

That ideal outcome is difficult to observe directly, so I will use a measurable proxy for this assignment.

The proxy is the difference between a page's observed CTR and the average CTR for pages in the same position tier.

I will call this:

`ctr_gap = page CTR - average CTR for the same position tier`

A strongly negative `ctr_gap` means the page receives fewer clicks than the typical page in a comparable position tier.

This is only a proxy. Low CTR can have multiple causes, including search intent mismatch, SERP competition, or other factors, so the proxy should be treated as decision-support rather than proof of a content-quality problem.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# Create a simple measurable proxy label when trend_direction is available

if "trend_direction" in df.columns:
    metric_df = df.copy()

    metric_df["proxy_label"] = (
        metric_df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

    print("Proxy label created successfully.")
    print()
    print("Proxy label counts:")
    print(metric_df["proxy_label"].value_counts())

    print()
    print("Proxy positive rate:")
    print(round(metric_df["proxy_label"].mean() * 100, 2), "%")

else:
    print("trend_direction is not available in this dataset.")
    print("Available columns:")
    print(df.columns.tolist())

Proxy label created successfully.

Proxy label counts:
proxy_label
1    16262
0    13738
Name: count, dtype: int64

Proxy positive rate:
54.21 %


### Success Metric: Precision@K

The main success metric is Precision@K.

I will use K = 20 as the primary review capacity.

Precision@20 measures how many of the top 20 ranked pages belong to the defined target or proxy group.

This metric matches the real decision because an editorial team has limited capacity and is likely to review only a small number of pages first.

A useful ranking system should concentrate relevant pages near the top of the review queue.

The metric evaluates how useful the ranking is for prioritization; it does not prove that a content refresh will cause performance improvement.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# Show the actual dataframe and demonstrate the unit of analysis

print("Unit of analysis: ONE CONTENT PAGE")
print("Number of rows:", len(df))
print()

# Show the first 10 rows
display(df.head(10))

Unit of analysis: ONE CONTENT PAGE
Number of rows: 30000



,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


### Unit of Analysis

The unit of analysis is one content page.

Therefore:

**One row = one content item/page.**

Each content page has measurable attributes such as impressions, clicks, CTR, average position, content age, freshness, engagement, and trend information.

The ranking system would assign an opportunity or priority score to each content page.

The resulting ranked list can then be used as a human review queue.

In [ ]:
print("Shape of the dataframe:", df.shape)
print()
print("Each row represents one content item/page.")

Shape of the dataframe: (30000, 44)

Each row represents one content item/page.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# Demonstrate a simple fixed-rule baseline

if "ctr" in df.columns:

    rule_df = df.copy()

    rule_df["ctr"] = pd.to_numeric(
        rule_df["ctr"],
        errors="coerce"
    )

    rule_df = rule_df.dropna(subset=["ctr"])

    # Example baseline threshold
    threshold = 0.5

    rule_df["fixed_rule_flag"] = (
        rule_df["ctr"] < threshold
    ).astype(int)

    print("Fixed-rule baseline")
    print("-------------------")
    print("Rule: CTR < 0.5%")
    print("Pages analysed:", len(rule_df))
    print("Pages flagged:", rule_df["fixed_rule_flag"].sum())

    print(
        "Percentage flagged:",
        round(
            rule_df["fixed_rule_flag"].mean() * 100,
            2
        ),
        "%"
    )

else:

    print("CTR column was not found.")
    print("Available columns:")
    print(df.columns.tolist())

Fixed-rule baseline
-------------------
Rule: CTR < 0.5%
Pages analysed: 30000
Pages flagged: 25750
Percentage flagged: 85.83 %


### Why ML Instead of a Fixed Rule?

A simple fixed rule could be something like:

"Refresh every page whose CTR is below a particular threshold."

Although this is easy to understand, one threshold does not consider the full context of a page.

For example, CTR can depend on search position, impressions, clicks, content age, freshness, engagement, and trend direction.

Two pages can have the same CTR but very different levels of search visibility and business opportunity.

An ML-based ranking approach can combine multiple signals and learn patterns from historical data rather than relying on only one manually selected threshold.

The fixed rule can still be used as a transparent baseline.

The ML output would remain decision-support: a human should review the highest-ranked pages before taking a content action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.